# 03 — Disagreement Exploration

This notebook explores *why* translation services disagree using the four-category typology from `explore_disagreements.py`:

| Category | Description |
|---|---|
| **MEASUREMENT_ARTEFACT** | Disagreement dissolves after normalization (Esperanto: one letter of grammatical agreement) |
| **STRUCTURAL_ABSENCE** | No community equivalent — model borrows or signals absence (Swahili) |
| **PRODUCTIVE_DISAGREEMENT** | Multiple legitimate in-language alternatives reflecting real debate (Arabic) |
| **TRANSMOGRIFICATION** | Confident fluent output untethered from community practice (Gothic) |

Input: `translated_terms/digital_humanities/evaluation/disagreement_analysis.csv`

In [1]:
import os
import sys
import ast
import pandas as pd
import altair as alt
from pathlib import Path

alt.data_transformers.enable('vegafusion')

sys.path.insert(0, str(Path('..').resolve()))
from scripts.utils import get_data_directory_path, read_csv_file, get_language_family
from scripts.exploration.explore_confidence_within_variant import load_variant_df
from scripts.exploration.explore_disagreements import run_disagreement_analysis, CATEGORIES

DATA_DIR   = get_data_directory_path()
TERM       = 'Digital Humanities'
TERM_SLUG  = TERM.lower().replace(' ', '_')
EVAL_DIR   = os.path.join(DATA_DIR, 'translated_terms', TERM_SLUG, 'evaluation')

CATEGORY_ORDER = [
    'STRUCTURAL_ABSENCE', 'TRANSMOGRIFICATION',
    'PRODUCTIVE_DISAGREEMENT', 'MEASUREMENT_ARTEFACT', 'NOT_APPLICABLE'
]
CATEGORY_COLOURS = {
    'STRUCTURAL_ABSENCE':      '#d62728',
    'TRANSMOGRIFICATION':      '#ff7f0e',
    'PRODUCTIVE_DISAGREEMENT': '#2ca02c',
    'MEASUREMENT_ARTEFACT':    '#1f77b4',
    'NOT_APPLICABLE':          '#aec7e8',
}
CATEGORY_LABELS = {
    'STRUCTURAL_ABSENCE':      'Structural Absence',
    'TRANSMOGRIFICATION':      'Transmogrification',
    'PRODUCTIVE_DISAGREEMENT': 'Productive Disagreement',
    'MEASUREMENT_ARTEFACT':    'Measurement Artefact',
    'NOT_APPLICABLE':          'All Agree',
}

## 3.1 Load Data

In [2]:
analysis_path = os.path.join(EVAL_DIR, 'disagreement_analysis.csv')
if not os.path.exists(analysis_path):
    print('Running explore_disagreements.py...')
    df = run_disagreement_analysis(DATA_DIR, [TERM])
else:
    df = read_csv_file(analysis_path)

df['category_label'] = df['category'].map(CATEGORY_LABELS)

# Load the minimal variant for full translation + rationale access
variant_df = load_variant_df(DATA_DIR, TERM_SLUG, 'minimal')

print(f'{len(df)} languages | {df["category"].value_counts().to_dict()}')
df.head(3)

857 languages | {'TRANSMOGRIFICATION': 446, 'STRUCTURAL_ABSENCE': 352, 'PRODUCTIVE_DISAGREEMENT': 32, 'MEASUREMENT_ARTEFACT': 14, 'NOT_APPLICABLE': 13}


,language_code,language_name,language_family,term_source,has_wikipedia,n_services_with_data,mean_rationale_similarity,category,evidence,loan_words_found,...,has_script_disagreement,scripts_found,script_per_service,script_signals,claude_rationale_snippet,openai_rationale_snippet,gemini_rationale_snippet,ollama_rationale_snippet,service_translations,category_label
0,aa,Afar,Afro-Asiatic languages,Digital Humanities,False,5,0.1688,TRANSMOGRIFICATION,No Wikipedia; 5 distinct outputs; absence sign...,False,...,False,['Latin'],"[('Google Translate', 'Latin'), ('OpenAI', 'La...",Google Translate:Latin; OpenAI:Latin; Claude:L...,"Combines 'Dijitaal' (digital, borrowed from En...",Combining 'Humaanitiyo' for 'Humanities' and '...,Afar lacks a direct equivalent for 'humanities...,"A combination of 'digital' and 'humanities', i...","{'Google Translate': 'Xiyitaal Sehdaytiino', '...",Transmogrification
1,ab,Abkhaz,Caucasian languages,Digital Humanities,False,5,0.1899,TRANSMOGRIFICATION,No Wikipedia; 5 distinct outputs; absence sign...,False,...,True,"['Cyrillic', 'Georgian']","[('Google Translate', 'Cyrillic'), ('OpenAI', ...",Google Translate:Cyrillic; OpenAI:Cyrillic; Cl...,Compound term combining 'Цифрақәа' (digital/nu...,The translated term captures 'digital' as 'Аци...,"The translation combines 'Ацифра' (digital), '...",Combination of Abkhazian words for 'digital' (...,{'Google Translate': 'Ацифратә гуманитартә ҵар...,Transmogrification
2,abq,Abaza,Caucasian languages,Digital Humanities,False,4,0.1933,TRANSMOGRIFICATION,No Wikipedia; 4 distinct outputs; absence sign...,False,...,True,"['Arabic', 'Cyrillic']","[('OpenAI', 'Cyrillic'), ('Claude', 'Cyrillic'...",OpenAI:Cyrillic; Claude:Cyrillic; Gemini:Cyril...,Abaza translation combining 'цифра' (digital) ...,"In Abaza, 'digital' can be translated as 'цифр...",This translation combines 'Акъымчӏара' (digita...,"Combining Abaza words 'فةڤ' (digital), 'خعونيد...","{'OpenAI': 'цифралтә гуманитартә нрацьхыл', 'C...",Transmogrification


## 3.2 Overview: Confidence vs Pairwise Translation Similarity

Scatter plot placing every language at the intersection of LLM exact-string confidence (x) and mean pairwise Levenshtein similarity across LLM service outputs (y). The two axes capture different failure modes: low confidence means the models disagree on which string is best; low similarity means the strings they produce are genuinely far apart. Together they separate four diagnostic zones.

In [3]:
import itertools
from scripts.exploration.explore_disagreements import edit_distance

# ── Mean pairwise Levenshtein similarity from LLM service translations ──
_LLM_SVCS = {'OpenAI', 'Claude', 'Gemini', 'First Ollama', 'Second Ollama'}

def _lev_sim(a: str, b: str) -> float:
    max_len = max(len(a), len(b))
    return 1.0 - edit_distance(a, b) / max_len if max_len else 1.0

def _mean_pairwise_lev(trans_str) -> float:
    try:
        d = ast.literal_eval(str(trans_str))
    except Exception:
        return float('nan')
    vals = [str(v).strip() for k, v in d.items()
            if k in _LLM_SVCS and v and str(v) not in ('nan', '', 'None')]
    if len(vals) < 2:
        return float('nan')
    sims = [_lev_sim(a, b) for a, b in itertools.combinations(vals, 2)]
    return sum(sims) / len(sims)

# ── Mean LLM confidence averaged across all variants ──
conf_path = os.path.join(EVAL_DIR, 'confidence_scores.csv')
conf_df = read_csv_file(conf_path)
conf_avg = (
    conf_df[conf_df['llm_total_services'] > 0]
    .groupby(['language_code', 'term_source'])['llm_confidence']
    .mean()
    .reset_index()
    .rename(columns={'llm_confidence': 'mean_llm_confidence'})
)

# ── Build scatter dataset ──
scatter_df = df.copy()
scatter_df['mean_pairwise_sim'] = scatter_df['service_translations'].apply(_mean_pairwise_lev)
scatter_df = scatter_df.merge(conf_avg, on=['language_code', 'term_source'], how='left')
scatter_df = scatter_df.dropna(subset=['mean_llm_confidence', 'mean_pairwise_sim'])

# Map disagreement categories to the four diagnostic case types
CASE_MAP = {
    'MEASUREMENT_ARTEFACT':    'false low (morphological variation)',
    'STRUCTURAL_ABSENCE':      'true fragmentation (absent concept)',
    'PRODUCTIVE_DISAGREEMENT': 'high confidence',
    'NOT_APPLICABLE':          'high confidence',
    'TRANSMOGRIFICATION':      'uncertain',
}
scatter_df['case_type'] = scatter_df['category'].map(CASE_MAP).fillna('uncertain')

CASE_COLOURS = {
    'false low (morphological variation)': '#f28e2b',
    'true fragmentation (absent concept)': '#e15759',
    'uncertain':                           '#b0b0b0',
    'high confidence':                     '#4e79a7',
}
CASE_ORDER = list(CASE_COLOURS.keys())

print(scatter_df['case_type'].value_counts().to_string())
print(f'\nConfidence  {scatter_df["mean_llm_confidence"].min():.2f} – {scatter_df["mean_llm_confidence"].max():.2f}')
print(f'Similarity  {scatter_df["mean_pairwise_sim"].min():.2f} – {scatter_df["mean_pairwise_sim"].max():.2f}')

case_type
uncertain                              446
true fragmentation (absent concept)    352
high confidence                         45
false low (morphological variation)     14

Confidence  0.25 – 1.00
Similarity  0.00 – 1.00


In [25]:
CONF_THRESH = 0.3
SIM_THRESH  = 0.5

n_cases    = scatter_df['case_type'].value_counts()
false_low_n = int(n_cases.get('false low (morphological variation)', 0))
frag_n      = int(n_cases.get('true fragmentation (absent concept)', 0))

points = alt.Chart(scatter_df).mark_circle(opacity=0.65, size=70).encode(
    x=alt.X('mean_llm_confidence:Q',
            scale=alt.Scale(domain=[0, 1.05]),
            title='LLM confidence (exact-string)'),
    y=alt.Y('mean_pairwise_sim:Q',
            scale=alt.Scale(domain=[0, 1.1]),
            title='Mean pairwise similarity (Levenshtein)'),
    color=alt.Color('case_type:N',
        scale=alt.Scale(domain=CASE_ORDER, range=list(CASE_COLOURS.values())),
        title='Case type'),
    tooltip=[
        'language_name:N', 'language_family:N', 'case_type:N',
        alt.Tooltip('mean_llm_confidence:Q', format='.3f'),
        alt.Tooltip('mean_pairwise_sim:Q', format='.3f'),
    ],
)

vline = alt.Chart(pd.DataFrame({'x': [CONF_THRESH]})).mark_rule(
    strokeDash=[6, 4], color='#888', opacity=0.5
).encode(x='x:Q')

hline = alt.Chart(pd.DataFrame({'y': [SIM_THRESH]})).mark_rule(
    strokeDash=[6, 4], color='#888', opacity=0.5
).encode(y='y:Q')

annot_data = pd.DataFrame([
    {'x': 0.02, 'y': 0.93, 'text': 'False low confidence (morphological variation)'},
    {'x': 0.02, 'y': 0.13, 'text': 'True fragmentation (absent concept)'},
    {'x': 0.33, 'y': 0.26, 'text': 'High confidence (settled translation)'},
])

annot = alt.Chart(annot_data).mark_text(
    align='left', fontStyle='italic', fontSize=10, color='#444'
).encode(
    x=alt.X('x:Q', scale=alt.Scale(domain=[0, 1.05])),
    y=alt.Y('y:Q', scale=alt.Scale(domain=[0, 1.1])),
    text='text:N',
)

(points + vline + hline ).properties(
    width=520, height=430,
    title=alt.TitleParams(
        'Confidence Score vs Pairwise Translation Similarity',
        subtitle=[
            f'Bottom-left: {frag_n} languages where every service produces a genuinely different output (absent concept).',
            f'Top-left: {false_low_n} languages scored as low-confidence but actually near-matches — a measurement artefact.',
        ],
        subtitleFontSize=11,
        subtitleColor='#666',
    ),
)

alt.LayerChart(...)

## 3.2 Category Distribution

In [5]:
counts = (
    df.groupby('category')
    .size()
    .reset_index(name='n')
    .assign(
        label=lambda d: d['category'].map(CATEGORY_LABELS),
        pct=lambda d: (d['n'] / d['n'].sum() * 100).round(1),
        pct_label=lambda d: d.apply(lambda r: f"{r['label']}\n{r['n']} ({r['pct']}%)", axis=1),
    )
)

donut = alt.Chart(counts).mark_arc(innerRadius=60, outerRadius=120).encode(
    theta=alt.Theta('n:Q'),
    color=alt.Color(
        'category:N',
        scale=alt.Scale(
            domain=list(CATEGORY_COLOURS.keys()),
            range=list(CATEGORY_COLOURS.values())
        ),
        legend=alt.Legend(title='Category', labelExpr=
            "{'STRUCTURAL_ABSENCE':'Structural Absence','TRANSMOGRIFICATION':'Transmogrification',"
            "'PRODUCTIVE_DISAGREEMENT':'Productive Disagreement','MEASUREMENT_ARTEFACT':'Measurement Artefact',"
            "'NOT_APPLICABLE':'All Agree'}[datum.label]"
        )
    ),
    tooltip=['label:N', 'n:Q', 'pct:Q']
).properties(width=300, height=300, title='Disagreement Category Distribution')

bar = alt.Chart(counts).mark_bar().encode(
    x=alt.X('n:Q', title='Languages'),
    y=alt.Y('category:N', sort='-x', title=None,
            axis=alt.Axis(labelExpr=
                "{'STRUCTURAL_ABSENCE':'Structural Absence','TRANSMOGRIFICATION':'Transmogrification',"
                "'PRODUCTIVE_DISAGREEMENT':'Productive Disagreement','MEASUREMENT_ARTEFACT':'Measurement Artefact',"
                "'NOT_APPLICABLE':'All Agree'}[datum.label]"
            )),
    color=alt.Color('category:N',
        scale=alt.Scale(domain=list(CATEGORY_COLOURS.keys()), range=list(CATEGORY_COLOURS.values())),
        legend=None),
    text=alt.Text('pct:Q', format='.1f'),
    tooltip=['label:N', 'n:Q', 'pct:Q']
).mark_bar() + alt.Chart(counts).mark_text(align='left', dx=4).encode(
    x=alt.X('n:Q'),
    y=alt.Y('category:N', sort='-x'),
    text=alt.Text('pct:Q', format='.1f')
)

(donut | bar).resolve_scale(color='independent')

alt.HConcatChart(...)

## 3.3 Category × Language Family

In [6]:
interesting_cats = ['STRUCTURAL_ABSENCE', 'TRANSMOGRIFICATION', 'PRODUCTIVE_DISAGREEMENT', 'MEASUREMENT_ARTEFACT']

family_cat = (
    df[df['category'].isin(interesting_cats)]
    .groupby(['language_family', 'category'])
    .size()
    .reset_index(name='n')
    .assign(label=lambda d: d['category'].map(CATEGORY_LABELS))
)

# Total per family for sorting
family_totals = family_cat.groupby('language_family')['n'].sum().reset_index(name='total')
family_cat = family_cat.merge(family_totals, on='language_family')

alt.Chart(family_cat).mark_rect().encode(
    x=alt.X('label:N', sort=list(CATEGORY_LABELS.values())[:4], title=None),
    y=alt.Y('language_family:N',
            sort=alt.EncodingSortField(field='total', order='descending'),
            title='Language Family'),
    color=alt.Color('n:Q',
        scale=alt.Scale(scheme='orangered'),
        title='Languages'),
    tooltip=['language_family:N', 'label:N', 'n:Q']
).properties(
    width=420, height=400,
    title='Disagreement Categories by Language Family'
)

alt.Chart(...)

## 3.4 Measurement Artefacts

Disagreements that dissolve after normalization — the scoring tool's limit, not a real divergence. These are useful as a calibration check: if the string-matching algorithm calls these "different", that tells us something about what we can and can't claim from exact-match scoring.

In [7]:
artefacts = df[df['category'] == 'MEASUREMENT_ARTEFACT'].copy()
print(f'{len(artefacts)} measurement artefacts')

# Show key columns
display(artefacts[[
    'language_code', 'language_name', 'language_family',
    'n_unique_raw', 'n_unique_normalized', 'max_edit_distance', 'evidence',
    'service_translations'
]].sort_values('max_edit_distance', ascending=False))

14 measurement artefacts


,language_code,language_name,language_family,n_unique_raw,n_unique_normalized,max_edit_distance,evidence,service_translations
121,cdo,Min Dong Chinese,Sino-Tibetan languages,2,2,2,Normalized to 2 unique value(s); max edit dist...,"{'OpenAI': '数字人文学', 'Claude': '數字人文', 'Gemini'..."
405,lad,Ladino / Judeo-Spanish,Indo-European languages,2,2,2,Normalized to 2 unique value(s); max edit dist...,"{'OpenAI': 'Humanidades Digitales', 'Claude': ..."
650,sco,Scots,Indo-European languages,2,2,2,Normalized to 2 unique value(s); max edit dist...,"{'OpenAI': 'Digital Humanities', 'Claude': 'De..."
43,ast,Asturian,Indo-European languages,2,2,1,Normalized to 2 unique value(s); max edit dist...,"{'OpenAI': 'Humanidaes Dixitales', 'Claude': '..."
71,bfy,Bagheli,Indo-European languages,2,2,1,Normalized to 2 unique value(s); max edit dist...,"{'OpenAI': 'डिजिटल मानविकी', 'Claude': 'डिजिटल..."
143,co,Corsican,Indo-European languages,2,2,1,Normalized to 2 unique value(s); max edit dist...,"{'Google Translate': 'Umanità Digitale', 'Ling..."
163,da,Danish,Indo-European languages,2,2,1,Normalized to 2 unique value(s); max edit dist...,"{'Google Translate': 'Digitale humaniora', 'Ea..."
274,hak,Hakka Chinese,Sino-Tibetan languages,2,2,1,Normalized to 2 unique value(s); max edit dist...,"{'OpenAI': '數位人文', 'Claude': '數位人文', 'Gemini':..."
518,nb,Norwegian Bokmål,Indo-European languages,2,2,1,Normalized to 2 unique value(s); max edit dist...,"{'Google Translate': 'Digital humaniora', 'Ope..."
536,nn,Norwegian Nynorsk,Indo-European languages,2,2,1,Normalized to 2 unique value(s); max edit dist...,"{'OpenAI': 'Digital humaniora', 'Claude': 'Dig..."


In [8]:
# Edit distance distribution for artefacts
artefacts_plot = artefacts.dropna(subset=['max_edit_distance']).copy()
artefacts_plot['max_edit_distance'] = artefacts_plot['max_edit_distance'].astype(int)

alt.Chart(artefacts_plot).mark_bar(color=CATEGORY_COLOURS['MEASUREMENT_ARTEFACT']).encode(
    x=alt.X('max_edit_distance:O', title='Max Edit Distance (normalized)'),
    y=alt.Y('count():Q', title='Languages'),
    tooltip=['max_edit_distance:O', 'count():Q']
).properties(
    width=300, height=200,
    title='Edit Distance Distribution — Measurement Artefacts'
)

alt.Chart(...)

## 3.5 Structural Absence

The concept has no community equivalent. The model signals this through loan words, explicit "no equivalent" rationale text, or near-zero coverage. This is the dominant pattern — and it's a finding about where DH as a practice hasn't reached, not a pipeline failure.

In [9]:
absent = df[df['category'] == 'STRUCTURAL_ABSENCE'].copy()
print(f'{len(absent)} structural absence cases')
print(f'  Loan word detected: {absent["loan_words_found"].sum()}')
print(f'  Absence signals in rationale: {(absent["absence_signals"].str.len() > 0).sum()}')
print(f'  Zero services with data: {(absent["n_services_with_data"] == 0).sum()}')

352 structural absence cases
  Loan word detected: 327
  Absence signals in rationale: 278
  Zero services with data: 0


In [10]:
# Absence sub-signals breakdown
absent_signals = (
    absent.assign(
        has_loan=absent['loan_words_found'],
        has_rationale_signal=absent['absence_signals'].str.len() > 0,
        no_data=absent['n_services_with_data'] == 0,
    )
)

signal_counts = pd.DataFrame({
    'signal': ['Loan word in translation', 'Absence keyword in rationale', 'No service data at all'],
    'n': [
        absent_signals['has_loan'].sum(),
        absent_signals['has_rationale_signal'].sum(),
        absent_signals['no_data'].sum(),
    ]
})

alt.Chart(signal_counts).mark_bar(color=CATEGORY_COLOURS['STRUCTURAL_ABSENCE']).encode(
    x=alt.X('n:Q', title='Languages'),
    y=alt.Y('signal:N', sort='-x', title=None),
    tooltip=['signal:N', 'n:Q']
).properties(width=400, height=160, title='Structural Absence — Evidence Signals')

alt.Chart(...)

In [11]:
# Structural absence by family (stacked: loan word vs rationale signal)
absent_family = (
    absent
    .assign(signal_type=lambda d: d.apply(
        lambda r: 'Loan word' if r['loan_words_found']
        else ('Rationale signal' if len(str(r['absence_signals'])) > 2
              else 'No data / other'),
        axis=1
    ))
    .groupby(['language_family', 'signal_type'])
    .size()
    .reset_index(name='n')
)

family_order = (
    absent_family.groupby('language_family')['n'].sum()
    .sort_values(ascending=False).index.tolist()
)

alt.Chart(absent_family).mark_bar().encode(
    x=alt.X('n:Q', title='Languages'),
    y=alt.Y('language_family:N', sort=family_order, title=None),
    color=alt.Color('signal_type:N',
        scale=alt.Scale(
            domain=['Loan word', 'Rationale signal', 'No data / other'],
            range=['#d62728', '#ff9896', '#ffcdd2']
        ),
        title='Evidence'
    ),
    tooltip=['language_family:N', 'signal_type:N', 'n:Q']
).properties(width=400, height=380, title='Structural Absence by Language Family')

alt.Chart(...)

In [12]:
# Case study: Tagalog (tl) — structural absence via loan word borrowing
# Google Translate returns 'Digital Humanities' unchanged; EasyNMT produces
# 'Digital na mga Tao' (Digital People); shows unadapted borrowing + garbled output
tl_row = df[df['language_code'] == 'tl'].iloc[0] if 'tl' in df['language_code'].values else None
if tl_row is not None:
    print(f'Tagalog (tl) — category: {tl_row["category"]}')
    print(f'Evidence: {tl_row["evidence"]}')
    print(f'Service translations:\n  {tl_row["service_translations"]}')
    print(f'Absence signals: {tl_row["absence_signals"]}')
    if variant_df is not None and 'tl' in variant_df['language_code'].values:
        tl_v = variant_df[variant_df['language_code'] == 'tl'].iloc[0]
        for svc in ['claude', 'openai', 'gemini', 'ollama']:
            rationale = tl_v.get(f'{svc}_translation_rationale', '')
            if rationale and str(rationale) != 'nan':
                print(f'\n{svc.upper()} rationale: {str(rationale)[:400]}')

Tagalog (tl) — category: STRUCTURAL_ABSENCE
Evidence: Loan word detected: True; absence signals: 3; services with data: 6
Service translations:
  {'Google Translate': 'Digital Humanities', 'EasyNMT': 'Digital na mga Tao', 'Lingvanex': 'Digital Humanities', 'OpenAI': 'Digital na Humanidades', 'Claude': 'Dihital na Humanidades', 'Gemini': 'Digital na Humanidades'}
Absence signals: Claude:loanword; Claude:transliterat; Gemini:borrowing

CLAUDE rationale: This is a direct transliteration-adaptation that combines 'Dihital' (the Filipinized form of 'Digital') with 'Humanidades' (the Spanish loanword commonly used in Tagalog academic contexts for 'Humanities'). This approach is consistent with how technical academic terms are typically handled in Tagalog.

OPENAI rationale: The term 'Digital Humanities' is translated as 'Digital na Humanidades' in Tagalog, as 'Digital' remains the same and 'Humanidades' is the Tagalog term for 'Humanities'. The use of 'na' connects the two words, conforming t

## 3.6 Productive Disagreement

Multiple legitimate in-language alternatives reflecting real scholarly debate. Wikipedia presence is the key signal — it means some community has adopted the term and there is something to disagree *about*. These are the most epistemically interesting cases: the pipeline surfaces debates that a single-output system would have buried.

In [13]:
productive = df[df['category'] == 'PRODUCTIVE_DISAGREEMENT'].copy()
print(f'{len(productive)} productive disagreement cases')
display(productive[[
    'language_code', 'language_name', 'language_family',
    'n_unique_raw', 'has_wikipedia', 'debate_signals', 'service_translations'
]].sort_values('n_unique_raw', ascending=False))

32 productive disagreement cases


,language_code,language_name,language_family,n_unique_raw,has_wikipedia,debate_signals,service_translations
854,zu,Zulu,Niger-Kordofanian languages,7,True,NaN,"{'Wikipedia': 'Ezezibhangqiwe zoLuntu', 'Googl..."
690,sr,Serbian,Indo-European languages,7,True,NaN,"{'Wikipedia': 'Дигитална хуманистика', 'Google..."
791,vi,Vietnamese,Austro-Asiatic languages,5,True,NaN,"{'Wikipedia': 'Nhân văn số', 'Google Translate..."
429,lmo,Lombard,Indo-European languages,5,True,NaN,"{'Wikipedia': 'Informatega umanistega', 'Googl..."
319,it,Italian,Indo-European languages,5,True,NaN,"{'Wikipedia': 'Informatica umanistica', 'Googl..."
299,hu,Hungarian,Uralic languages,4,True,NaN,"{'Wikipedia': 'Digitális bölcsészet', 'Google ..."
779,uk,Ukrainian,Indo-European languages,4,True,NaN,"{'Wikipedia': 'Цифрові гуманітарні науки', 'Go..."
715,ta,Tamil,Dravidian languages,4,True,NaN,"{'Wikipedia': 'எண்ணிம மனிதவியல்', 'Google Tran..."
666,sh,Serbo-Croatian,Indo-European languages,4,True,NaN,"{'Wikipedia': 'Digitalne humanističke nauke', ..."
30,ar,Arabic,Afro-Asiatic languages,4,True,NaN,"{'Wikipedia': 'إنسانيات رقمية', 'Google Transl..."


In [14]:
# Productive disagreement: how many unique translations per language?
prod_unique = (
    productive.groupby('n_unique_raw')
    .size()
    .reset_index(name='n_languages')
)

alt.Chart(prod_unique).mark_bar(color=CATEGORY_COLOURS['PRODUCTIVE_DISAGREEMENT']).encode(
    x=alt.X('n_unique_raw:O', title='Unique translation candidates'),
    y=alt.Y('n_languages:Q', title='Languages'),
    tooltip=['n_unique_raw:O', 'n_languages:Q']
).properties(
    width=300, height=200,
    title='Productive Disagreement — Candidate Count'
)

alt.Chart(...)

In [15]:
# Case study: Arabic
ar_row = df[df['language_code'] == 'ar'].iloc[0] if 'ar' in df['language_code'].values else None
if ar_row is not None:
    print(f'Arabic (ar) — category: {ar_row["category"]}')
    print(f'Evidence: {ar_row["evidence"]}')
    print(f'Service translations:\n  {ar_row["service_translations"]}')
    print(f'Debate signals: {ar_row["debate_signals"]}')
    if variant_df is not None and 'ar' in variant_df['language_code'].values:
        ar_v = variant_df[variant_df['language_code'] == 'ar'].iloc[0]
        for svc in ['claude', 'openai', 'gemini', 'ollama']:
            rationale = ar_v.get(f'{svc}_translation_rationale', '')
            if rationale and str(rationale) != 'nan':
                print(f'\n{svc.upper()} rationale: {str(rationale)[:400]}')

Arabic (ar) — category: PRODUCTIVE_DISAGREEMENT
Evidence: Wikipedia translation present; 4 distinct outputs; debate signals: 0
Service translations:
  {'Wikipedia': 'إنسانيات رقمية', 'Google Translate': 'العلوم الإنسانية الرقمية', 'EasyNMT': 'العلوم البشرية', 'Lingvanex': 'العلوم الإنسانية الرقمية', 'OpenAI': 'العلوم الإنسانية الرقمية', 'Claude': 'الإنسانيات الرقمية', 'Gemini': 'العلوم الإنسانية الرقمية', 'Ollama': 'العلوم الإنسانية الرقمية'}
Debate signals: nan

CLAUDE rationale: This is the standard and widely accepted translation of 'Digital Humanities' in Arabic academic discourse, combining 'الإنسانيات' (humanities) with 'الرقمية' (digital).

OPENAI rationale: The term 'العلوم الإنسانية الرقمية' directly translates the English term 'Digital Humanities' by combining 'العلوم الإنسانية' (Humanities) with 'الرقمية' (Digital), which is the standard translation widely accepted in Arabic-speaking academic circles.

GEMINI rationale: This is the standard and widely accepted translation in

## 3.7 Transmogrification

Confident, fluent, elaborate output untethered from any community practice. No Wikipedia translation, multiple different outputs, rationales are long and constructive (etymology, root words, compound formation). The rationales are the tell — they reveal a model that cannot say "I don't know" and instead reaches for plausible-seeming analogies.

In [16]:
transmog = df[df['category'] == 'TRANSMOGRIFICATION'].copy()
print(f'{len(transmog)} transmogrification cases')
print(f'  With construction signals: {(transmog["construction_signals"].str.len() > 2).sum()}')
print(f'  Mean unique translations: {transmog["n_unique_raw"].mean():.1f}')

446 transmogrification cases
  With construction signals: 350
  Mean unique translations: 4.0


In [17]:
# Construction keyword frequency across transmogrification cases
all_construction_signals = (
    transmog['construction_signals']
    .dropna()
    .str.split('; ')
    .explode()
    .str.strip()
    .loc[lambda s: s.str.len() > 0]
)

# Extract just the keyword part (after the colon)
kw_counts = (
    all_construction_signals
    .str.split(':', n=1).str[-1].str.strip()
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'keyword', 'construction_signals': 'count', 0: 'keyword', 'count': 'n'})
)
# Handle pandas version differences
if 'count' not in kw_counts.columns:
    kw_counts.columns = ['keyword', 'n']

if not kw_counts.empty:
    alt.Chart(kw_counts.head(15)).mark_bar(color=CATEGORY_COLOURS['TRANSMOGRIFICATION']).encode(
        x=alt.X('n:Q', title='Occurrences'),
        y=alt.Y('keyword:N', sort='-x', title=None),
        tooltip=['keyword:N', 'n:Q']
    ).properties(
        width=400, height=300,
        title='Construction Keywords in Transmogrification Rationales'
    )
else:
    print('No construction signals found in this variant — try running with --variant expert_persona')

In [18]:
# Number of unique translations per language (divergence depth)
transmog_unique = transmog.groupby('n_unique_raw').size().reset_index(name='n_languages')

alt.Chart(transmog_unique).mark_bar(color=CATEGORY_COLOURS['TRANSMOGRIFICATION']).encode(
    x=alt.X('n_unique_raw:O', title='Unique translation candidates'),
    y=alt.Y('n_languages:Q', title='Languages'),
    tooltip=['n_unique_raw:O', 'n_languages:Q']
).properties(
    width=300, height=200,
    title='Transmogrification — Candidate Divergence'
)

alt.Chart(...)

In [19]:
# Case study: Gothic (got) — the archetypal transmogrification case
# No Wikipedia, 5 LLM services all produce different elaborate constructions.
# Rationales mention 'loanword' while ALSO constructing novel Gothic terms —
# the model cannot say no and reaches for etymology instead.
got_row = df[df['language_code'] == 'got'].iloc[0] if 'got' in df['language_code'].values else None
if got_row is not None:
    print(f'Gothic (got) — category: {got_row["category"]}')
    print(f'Evidence: {got_row["evidence"]}')
    print(f'Service translations:\n  {got_row["service_translations"]}')
    print(f'Construction signals: {got_row["construction_signals"]}')
    if variant_df is not None and 'got' in variant_df['language_code'].values:
        got_v = variant_df[variant_df['language_code'] == 'got'].iloc[0]
        for svc in ['claude', 'openai', 'gemini', 'ollama']:
            rationale = got_v.get(f'{svc}_translation_rationale', '')
            if rationale and str(rationale) != 'nan':
                print(f'\n{svc.upper()} rationale:\n{str(rationale)[:600]}')

Gothic (got) — category: TRANSMOGRIFICATION
Evidence: No Wikipedia; 4 distinct outputs; absence signals: 2; construction signals: 8; script disagreement: Latin+Other
Service translations:
  {'OpenAI': '𐌳𐌹𐌲𐍉𐍄𐍃 𐌷𐌿𐌼𐌰𐌽𐌹𐌸𐌴𐌹𐍃', 'Claude': 'digitala manwiskodas', 'Gemini': '𐌳𐌹𐌲𐌹𐍄𐌰𐌻𐌰 𐌷𐌿𐌼𐌰𐌽𐌹𐍄𐌰𐍄𐌴𐌹𐍃', 'Ollama': 'Witodihs gahs'}
Construction signals: Claude:suffix; OpenAI:compound; OpenAI:reconstructed; OpenAI:construct; OpenAI:formation; Gemini:ancient; Gemini:morpholog; Ollama:combining

CLAUDE rationale:
Combines 'digitala' (digital, loan adaptation) with 'manwiskodas' (humanities, from 'manna' = human + '-iskods' suffix for abstract nouns/fields of study)

OPENAI rationale:
The translation uses reconstructed terms for 'digital' as '𐌳𐌹𐌲𐍉𐍄𐍃' (digots) related to numbers or calculations, and '𐌷𐌿𐌼𐌰𐌽𐌹𐌸𐌴𐌹𐍃' (humaitheis) from 'humanities,' indicating the use of humanities disciplines in the digital realm. The structure reflects typical compound word formations in Gothic.

GEMINI rationale:
Gothic lacks

## 3.8 Rationale Analysis

Compare rationale characteristics across categories — length, keyword density, construction signal rate. The hypothesis: transmogrification produces longer, more elaborate rationales as a signal of the model constructing rather than recalling.

In [20]:
# Attach rationale text from variant_df for length analysis
if variant_df is not None:
    rationale_cols = [c for c in variant_df.columns if c.endswith('_translation_rationale')]
    rat_sub = variant_df[['language_code', 'term_source'] + rationale_cols].copy()

    # Mean rationale length across services per language
    rat_sub['mean_rationale_length'] = rat_sub[rationale_cols].apply(
        lambda row: pd.Series([
            len(str(v)) for v in row if v and str(v) != 'nan'
        ]).mean(),
        axis=1
    )

    df_with_rat = df.merge(
        rat_sub[['language_code', 'term_source', 'mean_rationale_length']],
        on=['language_code', 'term_source'],
        how='left'
    )
    print(df_with_rat.groupby('category')['mean_rationale_length'].agg(['mean','median','count']).round(0))
else:
    df_with_rat = df.copy()
    print('No variant data loaded — skipping rationale length analysis')

                          mean  median  count
category                                     
MEASUREMENT_ARTEFACT     217.0   213.0     14
NOT_APPLICABLE           200.0   194.0     13
PRODUCTIVE_DISAGREEMENT  178.0   176.0     32
STRUCTURAL_ABSENCE       242.0   234.0    352
TRANSMOGRIFICATION       249.0   243.0    446


In [21]:
if 'mean_rationale_length' in df_with_rat.columns:
    rat_plot = df_with_rat.dropna(subset=['mean_rationale_length']).copy()
    rat_plot = rat_plot[rat_plot['category'].isin(['STRUCTURAL_ABSENCE','TRANSMOGRIFICATION','PRODUCTIVE_DISAGREEMENT'])]
    rat_plot['label'] = rat_plot['category'].map(CATEGORY_LABELS)

    alt.Chart(rat_plot).mark_boxplot(extent='min-max').encode(
        x=alt.X('mean_rationale_length:Q', title='Mean rationale length (chars)'),
        y=alt.Y('label:N', sort=None, title=None),
        color=alt.Color('category:N',
            scale=alt.Scale(domain=list(CATEGORY_COLOURS.keys()), range=list(CATEGORY_COLOURS.values())),
            legend=None)
    ).properties(
        width=450, height=200,
        title='Rationale Length by Disagreement Category'
    )

## 3.9 Rationale Similarity by Category

TF-IDF cosine similarity between service rationales, computed with character n-grams (3–5) so the metric is script-agnostic. High similarity means the models converged on the same *reasoning*; low similarity means the models are reaching for different framings even when they agreed on a translation.

Hypothesis: **PRODUCTIVE_DISAGREEMENT** rows should show high similarity (models reason from the same community evidence even while choosing different terms); **TRANSMOGRIFICATION** rows should show low similarity (each model invents its own construction).

In [22]:
from scripts.exploration.explore_disagreements import compute_rationale_similarity

# Compute similarity for each language using the loaded variant_df rationales
if variant_df is not None:
    rationale_cols_map = {
        'Claude':  'claude_translation_rationale',
        'OpenAI':  'openai_translation_rationale',
        'Gemini':  'gemini_translation_rationale',
        'Ollama':  'ollama_translation_rationale',
    }

    def _row_similarity(vrow):
        rats = {}
        for svc, col in rationale_cols_map.items():
            val = vrow.get(col)
            rats[svc] = str(val) if val and str(val) != 'nan' else None
        return compute_rationale_similarity(rats)

    sim_series = variant_df.apply(_row_similarity, axis=1)
    variant_df_with_sim = variant_df.copy()
    variant_df_with_sim['mean_rationale_similarity'] = sim_series

    # Drop pre-existing column to avoid _x/_y suffix collision on merge
    df_base = df.drop(columns=['mean_rationale_similarity'], errors='ignore')
    df_with_sim = df_base.merge(
        variant_df_with_sim[['language_code', 'term_source', 'mean_rationale_similarity']],
        on=['language_code', 'term_source'],
        how='left'
    )
    print(df_with_sim.groupby('category')['mean_rationale_similarity'].agg(['mean', 'median', 'count']).round(3))
else:
    df_with_sim = df.copy()
    print('No variant data — skipping similarity analysis')

                          mean  median  count
category                                     
MEASUREMENT_ARTEFACT     0.340   0.351     14
NOT_APPLICABLE           0.332   0.291     13
PRODUCTIVE_DISAGREEMENT  0.329   0.319     32
STRUCTURAL_ABSENCE       0.210   0.203    352
TRANSMOGRIFICATION       0.219   0.212    446


In [23]:
sim_plot = df_with_sim.dropna(subset=['mean_rationale_similarity']).copy()
focus_cats = ['STRUCTURAL_ABSENCE', 'TRANSMOGRIFICATION', 'PRODUCTIVE_DISAGREEMENT']
sim_plot = sim_plot[sim_plot['category'].isin(focus_cats)]
sim_plot['label'] = sim_plot['category'].map(CATEGORY_LABELS)

selection = alt.selection_point(fields=['category'], bind='legend')

_scale = alt.Scale(domain=list(CATEGORY_COLOURS.keys()), range=list(CATEGORY_COLOURS.values()))
_label_expr = (
    "{'STRUCTURAL_ABSENCE':'Structural Absence','TRANSMOGRIFICATION':'Transmogrification',"
    "'PRODUCTIVE_DISAGREEMENT':'Productive Disagreement'}[datum.label]"
)
sel_color = alt.Color('category:N', scale=_scale, title='Category',
    legend=alt.Legend(labelExpr=_label_expr, symbolType='circle',
                      orient='right', titleFontSize=12, labelFontSize=11))

alt.renderers.enable('default')

box = alt.Chart(sim_plot).mark_boxplot(extent='min-max').encode(
    x=alt.X('mean_rationale_similarity:Q', scale=alt.Scale(domain=[0, 1]),
            title='Mean pairwise rationale similarity (TF-IDF cosine)'),
    y=alt.Y('label:N', sort=None, title=None),
    color=sel_color,
    # opacity=alt.when(selection).then(alt.value(1)).otherwise(alt.value(0.2)),
    tooltip=['label:N', alt.Tooltip('mean_rationale_similarity:Q', format='.3f')],
).properties(
    width=450, height=180,
    title='Rationale Similarity by Disagreement Category'
)

points = alt.Chart(sim_plot).mark_circle(size=60).encode(
    x=alt.X('n_unique_raw:Q', title='Unique translation candidates'),
    y=alt.Y('mean_rationale_similarity:Q', scale=alt.Scale(domain=[0, 1]),
            title='Rationale similarity'),
    color=sel_color,
    # opacity=alt.when(selection).then(alt.value(0.8)).otherwise(alt.value(0.1)),
    tooltip=['language_name:N', 'label:N',
             alt.Tooltip('mean_rationale_similarity:Q', format='.3f'),
             'n_unique_raw:Q'],
).properties(
    width=420, height=250,
    title='Rationale Similarity vs Translation Divergence'
)

box & points

alt.VConcatChart(...)

## 3.10 Full Table — All Disagreement Cases

In [24]:
display_cols = [
    'language_code', 'language_name', 'language_family', 'category',
    'n_unique_raw', 'has_wikipedia', 'loan_words_found',
    'absence_signals', 'construction_signals', 'debate_signals',
    'evidence', 'service_translations'
]

pd.set_option('display.max_colwidth', 80)
display(
    df[display_cols]
    .sort_values(['category', 'language_family', 'language_name'])
    .reset_index(drop=True)
)

,language_code,language_name,language_family,category,n_unique_raw,has_wikipedia,loan_words_found,absence_signals,construction_signals,debate_signals,evidence,service_translations
0,zbl,Blissymbols,Artificial languages,MEASUREMENT_ARTEFACT,4,False,False,NaN,Ollama:combining,NaN,Normalized to 1 unique value(s); max edit distance 0 ≤ threshold 3,"{'OpenAI': '☼✉', 'Claude': '📱💻📚🔬👥', 'Gemini': '⠓⠥⠍⠁⠝⠔⠞⠽ ⠎⠞⠥⠙⠽ ⠺⠔⠞⠓ ⠙⠔⠛⠔⠞⠁⠇ ⠞..."
1,ast,Asturian,Indo-European languages,MEASUREMENT_ARTEFACT,2,False,False,NaN,NaN,NaN,Normalized to 2 unique value(s); max edit distance 1 ≤ threshold 3,"{'OpenAI': 'Humanidaes Dixitales', 'Claude': 'Humanidaes Dixitales', 'Gemini..."
2,bfy,Bagheli,Indo-European languages,MEASUREMENT_ARTEFACT,2,False,False,Claude:transliterat; OpenAI:transliterat; Gemini:loanword,Gemini:combining; Ollama:combining,NaN,Normalized to 2 unique value(s); max edit distance 1 ≤ threshold 2,"{'OpenAI': 'डिजिटल मानविकी', 'Claude': 'डिजिटल मानविकी', 'Gemini': 'डिजिटल म..."
3,co,Corsican,Indo-European languages,MEASUREMENT_ARTEFACT,2,False,True,NaN,Claude:combining; Ollama:combining,NaN,Normalized to 2 unique value(s); max edit distance 1 ≤ threshold 2,"{'Google Translate': 'Umanità Digitale', 'Lingvanex': 'Umanità digitale', 'O..."
4,da,Danish,Indo-European languages,MEASUREMENT_ARTEFACT,2,False,True,Gemini:loanword,NaN,NaN,Normalized to 2 unique value(s); max edit distance 1 ≤ threshold 2,"{'Google Translate': 'Digitale humaniora', 'EasyNMT': 'Digitale humaniora', ..."
...,...,...,...,...,...,...,...,...,...,...,...,...
852,vep,Veps,Uralic languages,TRANSMOGRIFICATION,4,False,False,NaN,Claude:compound; Claude:formation; Ollama:combining,NaN,No Wikipedia; 4 distinct outputs; absence signals: 0; construction signals: 3,"{'OpenAI': 'Digitaalsed humanitaatištudijad', 'Claude': 'Cifrine humanitarte..."
853,vot,Votic,Uralic languages,TRANSMOGRIFICATION,4,False,False,Claude:borrowing; OpenAI:borrowing; Gemini:borrowing; Ollama:borrowing,Claude:compound; Claude:construct; Claude:morpholog; OpenAI:combining,NaN,No Wikipedia; 4 distinct outputs; absence signals: 4; construction signals: 4,"{'OpenAI': 'Digitaalizet tiedosmit', 'Claude': 'Digitaalsõt humanitaarsõt ti..."
854,fiu-vro,Võro,Uralic languages,TRANSMOGRIFICATION,4,False,False,NaN,Claude:compound; Claude:combining; Ollama:combining,NaN,No Wikipedia; 4 distinct outputs; absence signals: 0; construction signals: 3,"{'OpenAI': 'digitaale umõndaus', 'Claude': 'Digitaal humanitaarteadusõq', 'G..."
855,vro,Võro,Uralic languages,TRANSMOGRIFICATION,4,False,False,Ollama:borrowed,Claude:compound; Claude:combining; Gemini:compound,NaN,No Wikipedia; 4 distinct outputs; absence signals: 1; construction signals: 3,"{'OpenAI': 'Digitaalse humanitaaria', 'Claude': 'Digitaalsõ humanitaartiidüs..."


## 3.11 Build Explorer Data

Run the standalone script to merge disagreement analysis, confidence stats, and parsed service translations into `disagreement_explorer_data.csv` for the HTML explorer:

```bash
python scripts/exploration/build_disagreement_explorer_data.py
# optional flags:
#   --term "Digital Humanities"
#   --output-dir path/to/dir
```

Then open `html_files/disagreement_explorer.html` in a browser and load the CSV.